In [ ]:
# 1. install pyspark library
!pip install pyspark

# 2. import theconcept proncipale
from pyspark.sql import SparkSession

# 3. make a new session spark
spark = SparkSession.builder \
    .appName("Football_Data_Engineering_Pipeline") \
    .master("local[*]") \
    .getOrCreate()

# 4. be sure the session working well
print("🚀 Spark Session is successfully created and ready to use!")
spark

🚀 Spark Session is successfully created and ready to use!


In [ ]:
# 1 READ THE RAW DATA
df_raw = spark.read.csv("/content/ensemble-de-donnees-6a0a3f2316332589898898.csv", header=True, inferSchema=True)

# SHOW THE SCHEMA OF THE RAW DATA
print("🔍 Raw Data Schema:")
df_raw.printSchema()

# 2. SELECT ONLY THE COLUMNS WE NEED
columns_to_keep = ['Match_ID', 'Div', 'Season', 'Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR']

# FILTER THE DATA TO KEEP ONLY THESE COLUMNS
df_selected = df_raw.select(*columns_to_keep)

# 3. RENAME THE FTR COLUMN TO FinalResult
df_cleaned = df_selected.withColumnRenamed("FTR", "FinalResult")

# 4. SHOW THE FIRST 5 ROWS TO CONFIRM THE OPERATION WAS SUCCESSFUL
print("✅ Cleaned Data (First 5 rows):")
df_cleaned.show(5)

🔍 Raw Data Schema:
root
 |-- Match_ID: integer (nullable = true)
 |-- Div: string (nullable = true)
 |-- Season: integer (nullable = true)
 |-- Date: date (nullable = true)
 |-- HomeTeam: string (nullable = true)
 |-- AwayTeam: string (nullable = true)
 |-- FTHG: integer (nullable = true)
 |-- FTAG: integer (nullable = true)
 |-- FTR: string (nullable = true)

✅ Cleaned Data (First 5 rows):
+--------+---+------+----------+-------------+--------------+----+----+-----------+
|Match_ID|Div|Season|      Date|     HomeTeam|      AwayTeam|FTHG|FTAG|FinalResult|
+--------+---+------+----------+-------------+--------------+----+----+-----------+
|       1| D2|  2009|2010-04-04|   Oberhausen|Kaiserslautern|   2|   1|          H|
|       2| D2|  2009|2009-11-01|  Munich 1860|Kaiserslautern|   0|   1|          A|
|       3| D2|  2009|2009-10-04|Frankfurt FSV|Kaiserslautern|   1|   1|          D|
|       4| D2|  2009|2010-02-21|Frankfurt FSV|     Karlsruhe|   2|   1|          H|
|       5| D2|  20

In [ ]:
# 1. import the necessary functions for creating indicator columns
from pyspark.sql.functions import col, when

# 2. CREATE THE INDICATOR COLUMNS
df_indicators = df_cleaned.withColumn(
    "HomeTeamWin", when(col("FinalResult") == "H", 1).otherwise(0)
).withColumn(
    "AwayTeamWin", when(col("FinalResult") == "A", 1).otherwise(0)
).withColumn(
    "GameTie", when(col("FinalResult") == "D", 1).otherwise(0)
)

# 3. SHOW THE RESULTS TO CONFIRM THE CONVERSION WAS SUCCESSFUL (WE SELECT ONLY SPECIFIC COLUMNS FOR EASIER READING)
print("✅ Indicator Columns Created Successfully:")
df_indicators.select("HomeTeam", "AwayTeam", "FinalResult", "HomeTeamWin", "AwayTeamWin", "GameTie").show(5)

✅ Indicator Columns Created Successfully:
+-------------+--------------+-----------+-----------+-----------+-------+
|     HomeTeam|      AwayTeam|FinalResult|HomeTeamWin|AwayTeamWin|GameTie|
+-------------+--------------+-----------+-----------+-----------+-------+
|   Oberhausen|Kaiserslautern|          H|          1|          0|      0|
|  Munich 1860|Kaiserslautern|          A|          0|          1|      0|
|Frankfurt FSV|Kaiserslautern|          D|          0|          0|      1|
|Frankfurt FSV|     Karlsruhe|          H|          1|          0|      0|
|        Ahlen|     Karlsruhe|          A|          0|          1|      0|
+-------------+--------------+-----------+-----------+-----------+-------+
only showing top 5 rows


In [ ]:
# 1. CREATE A FILTERED DATAFRAME FOR BUNDESLIGA (D1) SEASON 2000-2015
df_filtered = df_indicators.filter(
    (col("Div") == "D1") &
    (col("Season").between(2000, 2015))
)

# 2. SHOW THE NUMBER OF ROWS BEFORE AND AFTER FILTERING TO SEE THE DIFFERENCE (Data Validation)
print(f"📊 Total rows before filtering: {df_indicators.count()}")
print(f"📉 Total rows after filtering: {df_filtered.count()}")

# 3. SHOW A SAMPLE OF THE FILTERED DATA TO CONFIRM THE FILTERING WAS SUCCESSFUL
print("\n✅ Filtered Data (Bundesliga 2000-2015):")
df_filtered.select("Div", "Season", "HomeTeam", "AwayTeam", "FinalResult").show(5)

📊 Total rows before filtering: 24625
📉 Total rows after filtering: 4896

✅ Filtered Data (Bundesliga 2000-2015):
+---+------+-------------+----------+-----------+
|Div|Season|     HomeTeam|  AwayTeam|FinalResult|
+---+------+-------------+----------+-----------+
| D1|  2009|       Bochum|Leverkusen|          D|
| D1|  2009|Bayern Munich|Leverkusen|          D|
| D1|  2009|   M'gladbach|Leverkusen|          D|
| D1|  2009|        Mainz|Leverkusen|          D|
| D1|  2009|      Hamburg|Leverkusen|          D|
+---+------+-------------+----------+-----------+
only showing top 5 rows


In [ ]:
from pyspark.sql.functions import sum

# 1. AGGREGATE STATISTICS FOR HOME MATCHES
# For home matches, we group by Season and HomeTeam, and calculate the total wins, losses, ties, goals scored, and goals conceded for the home team.
df_home_matches = df_filtered.groupBy("Season", col("HomeTeam").alias("Team")).agg(
    sum("HomeTeamWin").alias("TotalHomeWin"),
    sum("AwayTeamWin").alias("TotalHomeLoss"), # 💡 IF THE HOME TEAM LOSES, THE AWAY TEAM WINS
    sum("GameTie").alias("TotalHomeTie"),
    sum("FTHG").alias("HomeScoredGoals"),      # FTHG = Full Time Home Goals
    sum("FTAG").alias("HomeAgainstGoals")      #    FTAG = Full Time Away Goals (goals conceded by the home team)
)

# 2. AGGREGATE STATISTICS FOR AWAY MATCHES
df_away_matches = df_filtered.groupBy("Season", col("AwayTeam").alias("Team")).agg(
    sum("AwayTeamWin").alias("TotalAwayWin"),
    sum("HomeTeamWin").alias("TotalAwayLoss"), # 💡 IF THE AWAY TEAM LOSES, THE HOME TEAM WINS
    sum("GameTie").alias("TotalAwayTie"),
    sum("FTAG").alias("AwayScoredGoals"),
    sum("FTHG").alias("AwayAgainstGoals")
)

# 3. MERGE THE HOME AND AWAY STATISTICS
# JOIN THE HOME AND AWAY DATAFRAMES ON SEASON AND TEAM TO GET A COMBINED VIEW OF EACH TEAM'S PERFORMANCE IN BOTH HOME AND AWAY MATCHES
df_merged = df_home_matches.join(
    df_away_matches,
    on=["Season", "Team"],
    how="inner"
)

# 4. SHOW THE RESULTS TO CONFIRM THE MERGING WAS SUCCESSFUL
print("✅ Merged Data (Home & Away combined):")
df_merged.show(5)

✅ Merged Data (Home & Away combined):
+------+-------------+------------+-------------+------------+---------------+----------------+------------+-------------+------------+---------------+----------------+
|Season|         Team|TotalHomeWin|TotalHomeLoss|TotalHomeTie|HomeScoredGoals|HomeAgainstGoals|TotalAwayWin|TotalAwayLoss|TotalAwayTie|AwayScoredGoals|AwayAgainstGoals|
+------+-------------+------------+-------------+------------+---------------+----------------+------------+-------------+------------+---------------+----------------+
|  2005|Bayern Munich|          14|            1|           2|             42|              14|           8|            2|           7|             25|              18|
|  2008|   M'gladbach|           5|            8|           4|             23|              27|           3|           11|           3|             16|              35|
|  2014|     Freiburg|           5|            6|           6|             21|              22|           2|         

In [ ]:
# 0. import the necessary functions for calculating advanced KPIs
from pyspark.sql.functions import round

# 1. aggregate the total wins, losses, ties, goals scored, and goals conceded for each team by summing the home and away statistics
df_kpis = df_merged.withColumn("Win", col("TotalHomeWin") + col("TotalAwayWin")) \
                   .withColumn("Loss", col("TotalHomeLoss") + col("TotalAwayLoss")) \
                   .withColumn("Tie", col("TotalHomeTie") + col("TotalAwayTie")) \
                   .withColumn("GoalsScored", col("HomeScoredGoals") + col("AwayScoredGoals")) \
                   .withColumn("GoalsAgainst", col("HomeAgainstGoals") + col("AwayAgainstGoals"))

# 2. calculate the total number of games played by each team (Total Games)
df_kpis = df_kpis.withColumn("TotalGames", col("Win") + col("Loss") + col("Tie"))

# 3. calculate the advanced KPIs
df_kpis = df_kpis.withColumn("GoalDifferentials", col("GoalsScored") - col("GoalsAgainst")) \
                 .withColumn("WinPercentage", round((col("Win") / col("TotalGames")) * 100, 2)) \
                 .withColumn("GoalsPerGame", round(col("GoalsScored") / col("TotalGames"), 2)) \
                 .withColumn("GoalsAgainstPerGame", round(col("GoalsAgainst") / col("TotalGames"), 2))

# 4. SHOW THE FINAL RESULTS IN AN ORGANIZED MANNER TO CONFIRM (WE SELECT ONLY THE IMPORTANT INDICATORS FOR DISPLAY)
print("🏆 Advanced KPIs Calculated Successfully:")
df_kpis.select("Season", "Team", "TotalGames", "WinPercentage", "GoalDifferentials", "GoalsPerGame").show(5)

🏆 Advanced KPIs Calculated Successfully:
+------+-------------+----------+-------------+-----------------+------------+
|Season|         Team|TotalGames|WinPercentage|GoalDifferentials|GoalsPerGame|
+------+-------------+----------+-------------+-----------------+------------+
|  2005|Bayern Munich|        34|        64.71|               35|        1.97|
|  2008|   M'gladbach|        34|        23.53|              -23|        1.15|
|  2014|     Freiburg|        34|        20.59|              -11|        1.06|
|  2015|    Wolfsburg|        34|        35.29|               -2|        1.38|
|  2006|      Cottbus|        34|        32.35|              -11|        1.12|
+------+-------------+----------+-------------+-----------------+------------+
only showing top 5 rows


In [ ]:
# 1. import the necessary functions for ranking teams within each season
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, desc

# 2. DEFINE A WINDOW SPECIFICATION TO RANK TEAMS WITHIN EACH SEASON
# we partition the data by season and order it descendingly by win percentage and goal differential
window_spec = Window.partitionBy("Season").orderBy(
    desc("WinPercentage"),
    desc("GoalDifferentials")
)

# 3. APPLY THE RANK FUNCTION TO THE WINDOW TO CREATE A TeamPosition COLUMN
df_ranked = df_kpis.withColumn("TeamPosition", rank().over(window_spec))

# 4. DISPLAY THE RESULTS: LET'S SEE THE RANKING OF TEAMS FOR SEASON 2015 TO CONFIRM THE CHAMPION OF THAT SEASON
print("🥇 Team Rankings for Season 2015:")
df_ranked.filter(col("Season") == 2015) \
         .select("Season", "TeamPosition", "Team", "WinPercentage", "GoalDifferentials") \
         .show(5)

🥇 Team Rankings for Season 2015:
+------+------------+-------------+-------------+-----------------+
|Season|TeamPosition|         Team|WinPercentage|GoalDifferentials|
+------+------------+-------------+-------------+-----------------+
|  2015|           1|Bayern Munich|        82.35|               63|
|  2015|           2|     Dortmund|        70.59|               48|
|  2015|           3|   Leverkusen|        52.94|               16|
|  2015|           4|   M'gladbach|         50.0|               17|
|  2015|           5|   Schalke 04|        44.12|                2|
+------+------------+-------------+-------------+-----------------+
only showing top 5 rows


In [ ]:
# 1. EXTRACT THE CHAMPIONS (TEAMS IN POSITION 1) FOR EACH SEASON
df_champions = df_ranked.filter(col("TeamPosition") == 1)

# show the champions of the last 5 seasons to confirm
print("🏆 Champions Extracted:")
df_champions.select("Season", "Team", "WinPercentage", "GoalDifferentials").orderBy(desc("Season")).show(5)

# 2. SAVE THE ENTIRE RANKED DATAFRAME IN PARQUET FORMAT, PARTITIONED BY SEASON
# we will save it in a folder named processed_data inside Colab
output_path_all = "processed_data/football_stats_partitioned"
df_ranked.write.mode("overwrite").partitionBy("Season").parquet(output_path_all)
print(f"✅ All teams stats saved partitioned by Season at: {output_path_all}")

# 3. SAVE THE CHAMPIONS' DATA ONLY IN PARQUET FORMAT
output_path_champions = "processed_data/football_top_teams"
df_champions.write.mode("overwrite").parquet(output_path_champions)
print(f"✅ Champions stats saved at: {output_path_champions}")

🏆 Champions Extracted:
+------+-------------+-------------+-----------------+
|Season|         Team|WinPercentage|GoalDifferentials|
+------+-------------+-------------+-----------------+
|  2015|Bayern Munich|        82.35|               63|
|  2014|Bayern Munich|        73.53|               62|
|  2013|Bayern Munich|        85.29|               71|
|  2012|Bayern Munich|        85.29|               80|
|  2011|     Dortmund|        73.53|               55|
+------+-------------+-------------+-----------------+
only showing top 5 rows
✅ All teams stats saved partitioned by Season at: processed_data/football_stats_partitioned
✅ Champions stats saved at: processed_data/football_top_teams


In [ ]:
# 1. SAVE THE ENTIRE RANKED DATAFRAME IN PARQUET FORMAT, PARTITIONED BY SEASON
output_path_all = "processed_data/football_stats_partitioned"

# 2. SAVE THE CHAMPIONS DATAFRAME IN PARQUET FORMAT
df_ranked.write.mode("overwrite").partitionBy("Season").parquet(output_path_all)

# 3. SAVE THE CHAMPIONS DATAFRAME IN PARQUET FORMAT
output_path_champions = "processed_data/football_top_teams"
df_champions.write.mode("overwrite").parquet(output_path_champions)

print("✅ Data saved successfully in Parquet format within Colab!")

✅ تمت عملية الحفظ بنجاح بصيغة Parquet مقسمة داخل Colab!


In [ ]:
# 1. ZIP THE ENTIRE processed_data FOLDER TO CREATE A SINGLE FILE
!zip -r football_parquet_files.zip processed_data/

# 2. CALL THE Colab LIBRARY TO DOWNLOAD THE FILE AUTOMATICALLY TO YOUR COMPUTER
from google.colab import files
files.download('football_parquet_files.zip')

  adding: processed_data/ (stored 0%)
  adding: processed_data/football_top_teams/ (stored 0%)
  adding: processed_data/football_top_teams/_SUCCESS (stored 0%)
  adding: processed_data/football_top_teams/._SUCCESS.crc (stored 0%)
  adding: processed_data/football_top_teams/.part-00000-a255da6f-5191-4328-9235-4a8d48beb228-c000.snappy.parquet.crc (stored 0%)
  adding: processed_data/football_top_teams/part-00000-a255da6f-5191-4328-9235-4a8d48beb228-c000.snappy.parquet (deflated 66%)
  adding: processed_data/football_stats_partitioned/ (stored 0%)
  adding: processed_data/football_stats_partitioned/_SUCCESS (stored 0%)
  adding: processed_data/football_stats_partitioned/Season=2014/ (stored 0%)
  adding: processed_data/football_stats_partitioned/Season=2014/part-00000-e9097541-269a-4c49-b5d7-1c053706f030.c000.snappy.parquet (deflated 65%)
  adding: processed_data/football_stats_partitioned/Season=2014/.part-00000-e9097541-269a-4c49-b5d7-1c053706f030.c000.snappy.parquet.crc (stored 0%)
  a

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# 1. import the necessary libraries for plotting
import matplotlib.pyplot as plt
import seaborn as sns

# 2. convert the champions' data from PySpark to Pandas
# we will order it by season first to make the plot chronological
pdf_champions = df_champions.orderBy("Season").toPandas()

# 3. prepare the style of the plots
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(3, 1, figsize=(12, 18)) # make 3 plots one above the other

# 📈 First: Win Percentage
sns.barplot(data=pdf_champions, x="Season", y="WinPercentage", ax=axes[0], hue="Team", dodge=False, palette="viridis")
axes[0].set_title("Win Percentage of Champions per Season (%)", fontsize=14, fontweight='bold')
axes[0].legend(loc='upper right', bbox_to_anchor=(1.15, 1)) # configure legend position and color position

# 📈 Second: Total Goals Scored
sns.lineplot(data=pdf_champions, x="Season", y="GoalsScored", ax=axes[1], marker="o", color="blue", linewidth=2.5)
axes[1].set_title("Total Goals Scored by Champions per Season", fontsize=14, fontweight='bold')

# 📈 Third: Goal Differential
sns.barplot(data=pdf_champions, x="Season", y="GoalDifferentials", ax=axes[2], hue="Team", dodge=False, palette="magma")
axes[2].set_title("Goal Differentials of Champions per Season", fontsize=14, fontweight='bold')
axes[2].legend(loc='upper right', bbox_to_anchor=(1.15, 1))

# 4. Adjust layout and show the plots
plt.tight_layout()
plt.show()